<a href="https://colab.research.google.com/github/senchiao/HRRR_plots/blob/main/gfs_aws.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install s3fs xarray cfgrib cartopy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.0/102.0 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 76.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [2]:
# Import necessary libraries
import s3fs
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import datetime

In [ ]:
def plot_gfs_200hpa_wind(date, cycle, forecast_hour, region=None):
    """
    Reads GFS 200 hPa wind data from AWS and plots it.

    Args:
        date (datetime.date): The date of the GFS model run.
        cycle (int): The forecast cycle (00, 06, 12, or 18 UTC).
        forecast_hour (int): The forecast hour (e.g., 0, 6, 12, etc.).
        region (list, optional): A list [lon_min, lon_max, lat_min, lat_max]
                                  to define a specific plotting region. Defaults to None (global).
    """

    # Initialize S3 file system
    fs = s3fs.S3FileSystem(anon=True) # anon=True for public buckets

    # Construct the S3 path for a GRIB2 file
    # For 0.25 degree resolution data:
    # Path: s3://noaa-gfs-bdp-pds/gfs.YYYYMMDD/CC/atmos/gfs.tCCz.pgrb2.0p25.fHHH.grib2
    s3_path = (f"s3://noaa-gfs-bdp-pds/gfs.{date.strftime('%Y%m%d')}/"
               f"{str(cycle).zfill(2)}/atmos/gfs.t{str(cycle).zfill(2)}z.pgrb2.0p25.f{str(forecast_hour).zfill(3)}")

    print(f"Attempting to open GFS data from: {s3_path}")

    try:
        with fs.open(s3_path, 'rb') as f:
            # Use xarray with cfgrib engine to open the GRIB2 file
            # We need U and V components of wind at 200 hPa (isobaricInhPa level)
            # 'filter_by_keys' is crucial to select specific levels/variables efficiently
            ds = xr.open_dataset(f, engine="cfgrib",
                                 backend_kwargs={'filter_by_keys': {'typeOfLevel': 'isobaricInhPa', 'level': 200}})

            print("\nDataset loaded. Variables found at 200 hPa:")
            print(list(ds.keys()))

            # Check if U and V wind components are available
            if 'u' in ds and 'v' in ds:
                u_wind = ds['u'].squeeze()
                v_wind = ds['v'].squeeze()

                # Calculate wind speed for contouring (optional, but good for visualizing jet streams)
                wind_speed = np.sqrt(u_wind**2 + v_wind**2)

                # Set up the plot
                fig = plt.figure(figsize=(12, 8))
                ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

                if region:
                    ax.set_extent(region, crs=ccrs.PlateCarree())
                else:
                    ax.set_global()

                ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
                ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
                ax.add_feature(cfeature.STATES, linewidth=0.3, edgecolor='gray') # For CONUS plots

                # Add gridlines with labels
                gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                                  linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
                gl.top_labels = False
                gl.right_labels = False

                # Plot wind speed as contours (optional)
                clevs = np.arange(10, 101, 10) # Wind speed contours from 10 to 100 m/s
                cbar_contour = ax.contourf(ds.longitude, ds.latitude, wind_speed,
                                           levels=clevs, cmap='viridis', extend='max', transform=ccrs.PlateCarree())
                plt.colorbar(cbar_contour, ax=ax, orientation='vertical', label='Wind Speed (m/s)')


                # Plot wind vectors (quivers or barbs)
                # To avoid plotting too many arrows, we can thin the data
                skip = 10 # Plot every 10th data point
                ax.quiver(ds.longitude[::skip], ds.latitude[::skip],
                          u_wind[::skip, ::skip], v_wind[::skip, ::skip],
                          color='black', transform=ccrs.PlateCarree(),
                          scale=1000, width=0.002, headwidth=3, headlength=4, alpha=0.7)

                ax.set_title(f'GFS 200 hPa Wind Forecast\n'
                             f'Run: {date.strftime("%Y-%m-%d")} {str(cycle).zfill(2)}Z, '
                             f'Forecast: +{str(forecast_hour).zfill(3)}h')

                plt.tight_layout()
                plt.show()

            else:
                print("Error: 'u' and 'v' wind components not found at 200 hPa.")

    except FileNotFoundError:
        print(f"Error: File not found at {s3_path}. Please check the path and date/cycle/forecast hour.")
    except Exception as e:
        print(f"An error occurred: {e}")

# --- Example Usage ---

if __name__ == "__main__":
    # Define the date and forecast
    current_date = datetime.date(2026, 8, 4) # YYYY, M, D
    current_cycle = 12 # 00, 06, 12, or 18 UTC
    forecast_time = 0 # Forecast hour, e.g., 0 for analysis, 6 for 6-hour forecast

    # Example 1: Plot global 200 hPa wind
    plot_gfs_200hpa_wind(current_date, current_cycle, forecast_time)

    # Example 2: Plot 200 hPa wind over a specific region (e.g., North America)
    # [lon_min, lon_max, lat_min, lat_max]
    north_america_region = [-130, -50, 20, 60]
    plot_gfs_200hpa_wind(current_date, current_cycle, forecast_time, region=north_america_region)

    # Example 3: Try a different forecast hour (e.g., 24-hour forecast)
    # plot_gfs_200hpa_wind(current_date, current_cycle, 24, region=north_america_region)

Attempting to open GFS data from: s3://noaa-gfs-bdp-pds/gfs.20260804/12/atmos/gfs.t12z.pgrb2.0p25.f000
